simple_rag_project/
│
├── notebook/
│   └── rag_project.ipynb
│
├── data/
│   └── sample.pdf
│
├── requirements.txt
│
└── .env

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

import os
import google.generativeai as genai
from dotenv import load_dotenv

C:\Users\Lenovo\Desktop\BCA-7\Lab\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11840\1489502746.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
loader = PyPDFLoader("../data/sample.pdf")
documents = loader.load()

In [3]:
print(documents[0].page_content)

Tribhuvan  University  is  the  oldest  and  largest  university  in  Nepal.  It  was  established  in  1959  and  named  
after
 
King
 
Tribhuvan.
 
The
 
university
 
was
 
founded
 
to
 
provide
 
higher
 
education
 
opportunities
 
within
 
Nepal,
 
as
 
many
 
students
 
previously
 
had
 
to
 
travel
 
abroad
 
for
 
advanced
 
studies.
 
Today,
 
it
 
plays
 
a
 
major
 
role
 
in
 
shaping
 
the
 
country’s
 
educational
 
system
 
and
 
producing
 
skilled
 
professionals
 
in
 
various
 
fields.
 
The  main  campus  of  Tribhuvan  University  is  located  in  Kirtipur,  near  Kathmandu.  The  university  has  a  
vast
 
academic
 
network
 
that
 
includes
 
constituent
 
campuses,
 
affiliated
 
colleges,
 
research
 
centers,
 
and
 
central
 
departments
 
spread
 
across
 
Nepal.
 
It
 
offers
 
programs
 
in
 
science,
 
management,
 
engineering,
 
medicine,
 
humanities,
 
education,
 
and
 
many
 
other
 
disciplines.
 
Tribhuvan  University  is  known  for  serving

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = text_splitter.split_documents(documents)

In [5]:
print(chunks[0].page_content)

Tribhuvan  University  is  the  oldest  and  largest  university  in  Nepal.  It  was  established


In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3704.90it/s]


In [9]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [10]:
vectorstore.save_local("faiss_index")

In [11]:
query = "When was TU established?"

results = vectorstore.similarity_search(query, k=3)

In [12]:
for i, doc in enumerate(results):
    print(f"Chunk {i+1}")
    print(doc.page_content)
    print("-" * 50)

Chunk 1
was  established  in  1959  and  named
--------------------------------------------------
Chunk 2
after
 
King
 
Tribhuvan.
 
The
 
university
 
was
 
founded
 
to
 
provide
 
higher
 
education
--------------------------------------------------
Chunk 3
development
 
throughout
 
the
 
nation.
--------------------------------------------------


In [13]:
load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [14]:
for model in genai.list_models():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [15]:
model = genai.GenerativeModel("models/gemini-2.5-flash")

In [16]:
query = input("Ask a question: ")

Ask a question:  Why was TU established?


In [17]:
retrieved_docs = vectorstore.similarity_search(query, k=3)

In [18]:
context = ", ".join([
    doc.page_content for doc in retrieved_docs
])

print(context)

was  established  in  1959  and  named, helped
 
in
 
promoting
 
research,
 
cultural
 
preservation,
 
and
 
academic
 
development, development
 
throughout
 
the
 
nation.


In [19]:
prompt = f"""
Answer the question only using the context below.

Context:
{context}

Question:
{query}
"""

In [20]:
print(prompt)


Answer the question only using the context below.

Context:
was  established  in  1959  and  named, helped
 
in
 
promoting
 
research,
 
cultural
 
preservation,
 
and
 
academic
 
development, development
 
throughout
 
the
 
nation.

Question:
Why was TU established?



In [21]:
response = model.generate_content(prompt)

In [22]:
print(response.text)

To help in promoting research, cultural preservation, and academic development throughout the nation.
